In [25]:
from mava.networks.retention import MultiScaleRetention
from omegaconf import DictConfig
import jax
import jax.numpy as jnp
import copy

# jax.config.update("jax_enable_x64", True)

bsz = 16
num_agents = 4
obs_dim = 11
num_time_steps = 100
seq_len = num_agents * num_time_steps

retnet_embed_dim = 32
retnet_num_heads = 2

In [26]:
memory_config = DictConfig(
    {
        "type": "rec_sable",
        "decay_scaling_factor": 0.3,
        "timestep_positional_encoding": True,
        "timestep_chunk_size": None,
    }
)

decay_kappas = 1 - jnp.exp(jnp.linspace(jnp.log(1 / 32), jnp.log(1 / 512), retnet_num_heads))
decay_kappas *= memory_config.decay_scaling_factor
decay_kappas = decay_kappas[None, :, None, None]

In [27]:
msr = MultiScaleRetention(
    embed_dim=retnet_embed_dim,
    n_head=retnet_num_heads,
    n_agents=num_agents,
    memory_config=memory_config,
    masked=False,
    decay_scaling_factor=memory_config.decay_scaling_factor,
)

In [28]:
key = jax.random.PRNGKey(0)
key, subkey = jax.random.split(key)

obs = jax.random.normal(subkey, (bsz, seq_len, retnet_embed_dim))

# assuming no resets
dones = jnp.zeros((bsz, seq_len), dtype=bool)

init_hstate = jnp.zeros((bsz, retnet_num_heads, retnet_embed_dim//retnet_num_heads, retnet_embed_dim//retnet_num_heads))
step_counts = jnp.arange(num_time_steps)
step_counts = step_counts[None, ...].repeat(bsz, axis=0)[..., None].repeat(num_agents, axis=-1)
step_counts = step_counts.reshape(bsz, seq_len)

In [29]:
key, init_key = jax.random.split(key)
params = msr.init(
    init_key,
    obs,
    obs,
    obs,
    init_hstate,
    dones,
    step_counts,
)

In [30]:
hstate = copy.deepcopy(init_hstate)
act_output = []


# for the decoder we use the chunkwise
for step in range(num_time_steps):

    # todo: reset later
    hstate = hstate * decay_kappas
    obs_i = obs[:, step*num_agents:(step+1)*num_agents, ...]
    dones_i = dones[:, step*num_agents:(step+1)*num_agents]
    step_counts_i = step_counts[:, step*num_agents:(step+1)*num_agents]

    out, hstate = msr.apply(params, obs_i, obs_i, obs_i, hstate, step_counts_i, method="recurrent")
    act_output.append(out)

In [31]:
act_output = jnp.concatenate(act_output, axis=1)

In [32]:
act_output.shape

(16, 400, 32)

In [33]:
hstate = copy.deepcopy(init_hstate)
train_out, _ = msr.apply(params, obs, obs, obs, hstate, dones, step_counts)

In [34]:
train_out.shape

(16, 400, 32)

In [35]:
total_error = jnp.mean(jnp.abs(train_out - act_output))
total_error

Array(6.5766194e-06, dtype=float32)

In [36]:
jnp.abs(train_out - act_output)

Array([[[3.36952507e-06, 3.07336450e-06, 1.11982226e-05, ...,
         1.53761357e-06, 3.13483179e-06, 1.90995634e-05],
        [2.98535451e-06, 1.11497939e-05, 7.54371285e-08, ...,
         1.26473606e-06, 1.07325613e-05, 1.53314322e-05],
        [8.27480108e-07, 9.88878310e-06, 3.66885215e-05, ...,
         3.98466364e-05, 3.90037894e-06, 7.28853047e-06],
        ...,
        [1.21723861e-06, 1.07660890e-06, 6.24451786e-07, ...,
         1.88359991e-06, 9.14093107e-07, 1.45751983e-07],
        [5.17242006e-06, 1.25039369e-05, 1.78515911e-05, ...,
         8.52765515e-06, 6.70552254e-07, 6.74137846e-06],
        [4.03262675e-06, 4.10713255e-06, 1.66334212e-06, ...,
         3.74019146e-06, 2.30502337e-07, 7.69970939e-06]],

       [[5.58421016e-06, 2.63168477e-06, 1.07008964e-05, ...,
         1.18725002e-05, 8.81217420e-06, 4.37628478e-06],
        [8.36490653e-06, 3.12947668e-06, 1.07632950e-05, ...,
         7.96373934e-06, 4.64729965e-06, 5.71832061e-07],
        [4.59374860e-06, 